# L2 matchups with PACE data -- RRS

* Create a plan for files to use `pc.plan()`
* Print the plan to check it `print(plan.summary())`
* Do the plan and get matchups `pc.matchup(plan, geometry="swath")`

## Prerequisite -- Login to EarthData

The examples here use NASA EarthData and you need to have an account with EarthData. Make sure you can login.

In [1]:
import earthaccess
import xoak
earthaccess.login()

## Here are the level 2 datasets

In [6]:
import earthaccess
results = earthaccess.search_datasets(instrument="oci")

short_names = [
    item.summary()["short-name"]
    for item in results
    if "L2" in item.summary()["short-name"]
]

print(short_names)

['PACE_OCI_L2_UVAI_UAA_NRT', 'PACE_OCI_L2_UVAI_UAA', 'PACE_OCI_L2_AER_UAA_NRT', 'PACE_OCI_L2_AER_UAA', 'PACE_OCI_L2_AOP_NRT', 'PACE_OCI_L2_AOP', 'PACE_OCI_L2_CLOUD_MASK_NRT', 'PACE_OCI_L2_CLOUD_MASK', 'PACE_OCI_L2_CLOUD_NRT', 'PACE_OCI_L2_CLOUD', 'PACE_OCI_L2_LANDVI_NRT', 'PACE_OCI_L2_LANDVI', 'PACE_OCI_L2_BGC_NRT', 'PACE_OCI_L2_BGC', 'PACE_OCI_L2_IOP_NRT', 'PACE_OCI_L2_IOP', 'PACE_OCI_L2_PAR_NRT', 'PACE_OCI_L2_PAR', 'PACE_OCI_L2_SFREFL_NRT', 'PACE_OCI_L2_SFREFL', 'PACE_OCI_L2_TRGAS_NRT', 'PACE_OCI_L2_TRGAS']


## Load some points

In [1]:
from pathlib import Path
import earthaccess
import point_collocation as pc
import pandas as pd

HERE = Path.cwd()
POINTS_CSV = HERE / "fixtures" / "points.csv"
df_points = pd.read_csv(POINTS_CSV)  # lat, lon, date columns
print(len(df_points))
df_points.head()

595


,lat,lon,date
0,27.3835,-82.7375,2024-06-13
1,27.1190,-82.7125,2024-06-14
2,26.9435,-82.8170,2024-06-14
3,26.6875,-82.8065,2024-06-14
4,26.6675,-82.6455,2024-06-14


## Get a plan for matchups from PACE data

In [2]:
%%time
# bounding_box = (lon_min, lat_min, lon_max, lat_max)
import point_collocation as pc
plan = pc.plan(
    df_points[0:100], # -82.7375, 27.3835	
    data_source="earthaccess",
    source_kwargs={
        "short_name": "PACE_OCI_L2_AOP",
    },
    time_buffer="12h"
)

CPU times: user 336 ms, sys: 19.5 ms, total: 356 ms
Wall time: 2.03 s


In [4]:
plan.summary()

Plan: 100 points → 24 unique granule(s)
  Points with 0 matches : 0
  Points with >1 matches: 20
  Time buffer: 0 days 12:00:00

First 5 point(s):
  [0] lat=27.3835, lon=-82.7375, time=2024-06-13 12:00:00: 2 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240613T171620.L2.OC_AOP.V3_1.nc
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240613T184939.L2.OC_AOP.V3_1.nc
  [1] lat=27.1190, lon=-82.7125, time=2024-06-14 12:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240614T175104.L2.OC_AOP.V3_1.nc
  [2] lat=26.9435, lon=-82.8170, time=2024-06-14 12:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240614T175104.L2.OC_AOP.V3_1.nc
  [3] lat=26.6875, lon=-82.8065, time=2024-06-14 12:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240614T175104.L2.OC_AOP.V3

In [5]:
plan.show_variables(geometry="swath")

geometry     : 'swath'
open_method  : 'datatree-merge'
Dimensions : {'number_of_bands': 286, 'number_of_reflective_bands': 286, 'wavelength_3d': 172, 'number_of_lines': 1710, 'pixels_per_line': 1272}
Variables  : ['wavelength', 'vcal_gain', 'vcal_offset', 'F0', 'aw', 'bbw', 'k_oz', 'k_no2', 'Tau_r', 'year', 'day', 'msec', 'time', 'detnum', 'mside', 'slon', 'clon', 'elon', 'slat', 'clat', 'elat', 'csol_z', 'Rrs', 'Rrs_unc', 'aot_865', 'angstrom', 'avw', 'nflh', 'l2_flags', 'longitude', 'latitude', 'tilt']

Geolocation: ('longitude', 'latitude') — lon dims=('number_of_lines', 'pixels_per_line'), lat dims=('number_of_lines', 'pixels_per_line')

DataTree groups (detail):
  /
    Dimensions : {}
    Variables  : []
  /sensor_band_parameters
    Dimensions : {'number_of_bands': 286, 'number_of_reflective_bands': 286, 'wavelength_3d': 172}
    Variables  : ['wavelength', 'vcal_gain', 'vcal_offset', 'F0', 'aw', 'bbw', 'k_oz', 'k_no2', 'Tau_r']
  /scan_line_attributes
    Dimensions : {'number_

## Get the matchups using that plan

In [3]:
%%time
res = pc.matchup(plan[0:5], geometry="swath", variables=["Rrs"])
res

CPU times: user 10.8 s, sys: 806 ms, total: 11.7 s
Wall time: 18.8 s


,lat,lon,time,granule_id,Rrs_346,Rrs_348,Rrs_351,Rrs_353,Rrs_356,Rrs_358,...,Rrs_706,Rrs_707,Rrs_708,Rrs_709,Rrs_711,Rrs_712,Rrs_713,Rrs_714,Rrs_717,Rrs_719
0,27.3835,-82.7375,2024-06-13 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,27.3835,-82.7375,2024-06-13 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,27.1190,-82.7125,2024-06-14 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,0.01299,0.012946,0.013148,0.013172,0.012918,0.012968,...,0.000238,0.000228,0.000198,0.000194,0.000186,0.000172,0.000152,0.000122,0.000108,0.000094
3,26.9435,-82.8170,2024-06-14 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,26.6875,-82.8065,2024-06-14 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,26.6675,-82.6455,2024-06-14 12:00:00,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
